# Orientation Bug Replication

Creates synthetic ellipse polygons at **known** angles and runs them through both orientation paths:

1. **`fitEllipse`** (scikit-image `EllipseModel`) → returns `theta` in radians
2. **`boulder_row`** (Shapely minimum rotated rectangle → `azimuth()`) → returns `angle`/`angle180`

For each input angle we compare what the code returns vs. what it should return.
If a column is constant regardless of input → that path contains the bug.

In [ ]:
import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import torch
import torch._utils
import sys
if not hasattr(torch, "_utils"):
    torch._utils = sys.modules["torch._utils"]

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely import segmentize
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

from sahi import AutoDetectionModel
from YOLOv8BeyondEarth.predict import get_sliced_predictionfast
from rastertools_BOULDERING import raster, metadata as raster_metadata, crs as raster_crs
from shptools_BOULDERING import shp, geometry as shp_geom, geomorph as shp_geomorph, annotations as shp_anno, metrics as shp_metrics
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row, azimuth

## Helper: generate a synthetic ellipse polygon at a known angle

In [ ]:
def make_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=200):
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t)
    y = b * np.sin(t)
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))


def make_noisy_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=40, noise_scale=0.05):
    rng = np.random.default_rng(seed=42)
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    noise = rng.normal(0, noise_scale * b, size=n_pts)
    r = 1.0 + noise
    x = a * np.cos(t) * r
    y = b * np.sin(t) * r
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))

In [ ]:
A   = 15.0
B   = 10.0
CX  = 500.0
CY  = 500.0
RES = 1.0

test_angles = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165]

---
## Part 1: Raw paths (independent)

In [ ]:
results = []
for deg in test_angles:
    poly = make_ellipse_polygon(A, B, deg, cx=CX, cy=CY)
    row  = pd.Series({"geometry": poly})
    try:
        _, a_fit, b_fit, theta_rad = fitEllipse(row)
        theta_deg = np.degrees(theta_rad)
    except Exception as e:
        theta_deg, a_fit, b_fit = None, None, None
        print(f"fitEllipse failed at {deg}°: {e}")
    try:
        mrr_row = pd.Series({"geometry": poly.minimum_rotated_rectangle})
        (_, _, _, _, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
    except Exception as e:
        angle_default, angle360, angle180 = None, None, None
        print(f"boulder_row failed at {deg}°: {e}")
    results.append({"input_deg": deg, "fitEllipse_theta_deg": theta_deg,
                    "fitEllipse_a": a_fit, "fitEllipse_b": b_fit,
                    "mrr_angle_default": angle_default, "mrr_angle180": angle180})

df = pd.DataFrame(results)
df

In [ ]:
expected_math = [d % 180 for d in test_angles]
# angle180 uses geographic convention (CW from North), so expected = (90 - theta) mod 180
expected_geo  = [((90 - d) % 180) or 180 for d in test_angles]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(test_angles, expected_math, 'k--', label='expected (math: input mod 180)')
axes[0].plot(test_angles, df["fitEllipse_theta_deg"] % 180, 'ro-', label='fitEllipse theta')
axes[0].set_title("Path 1: fitEllipse (EllipseModel)\nmath convention: CCW from East")
axes[0].legend(); axes[0].grid(True)
axes[1].plot(test_angles, expected_geo, 'k--', label='expected (geographic: (90°−input) mod 180)')
axes[1].plot(test_angles, df["mrr_angle180"], 'bs-', label='MRR angle180')
axes[1].set_title("Path 2: boulder_row (MRR + azimuth)\ngeographic convention: CW from North")
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.savefig("orientation_part1_raw_paths.png", dpi=150); plt.show()

---
## Part 2: Full post-processing pipeline (chained)

```
raw boulder polygon → segmentize → fitEllipse → MRR → boulder_row → final angle
```

In [ ]:
def full_pipeline(poly, res=RES):
    poly_seg = segmentize(poly, res)
    row_seg  = pd.Series({"geometry": poly_seg})
    ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
    theta_deg = np.degrees(theta_rad)
    mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
    (_, _, long_axis, short_axis, _, angle_default, angle360, angle180) = boulder_row(mrr_row)
    return {"fitEllipse_theta_deg": theta_deg, "fitEllipse_a": a_fit, "fitEllipse_b": b_fit,
            "final_angle_default": angle_default, "final_angle180": angle180,
            "aspect_ratio": long_axis / short_axis if short_axis > 0 else None}

In [ ]:
results_clean, results_noisy = [], []
for deg in test_angles:
    for results_list, poly_fn in [(results_clean, make_ellipse_polygon),
                                   (results_noisy, make_noisy_ellipse_polygon)]:
        poly = poly_fn(A, B, deg, cx=CX, cy=CY)
        try:
            r = full_pipeline(poly)
        except Exception as e:
            print(f"Pipeline failed at {deg}°: {e}")
            r = {k: None for k in ["fitEllipse_theta_deg","fitEllipse_a","fitEllipse_b",
                                    "final_angle_default","final_angle180","aspect_ratio"]}
        results_list.append({"input_deg": deg, **r})

df_clean = pd.DataFrame(results_clean)
df_noisy = pd.DataFrame(results_noisy)
print("=== Clean ==="); print(df_clean.to_string())
print("\n=== Noisy ==="); print(df_noisy.to_string())

In [ ]:
expected_math = [d % 180 for d in test_angles]
expected_geo  = [((90 - d) % 180) or 180 for d in test_angles]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for row_i, (df_i, label) in enumerate([(df_clean, "Clean"), (df_noisy, "Noisy")]):
    for col_i, (col, title) in enumerate([("fitEllipse_theta_deg", "fitEllipse theta"),
                                           ("final_angle180", "final angle180")]):
        exp = expected_math if col == "fitEllipse_theta_deg" else expected_geo
        exp_label = "expected (math: input mod 180)" if col == "fitEllipse_theta_deg" else "expected (geographic: (90°−input) mod 180)"
        axes[row_i, col_i].plot(test_angles, exp, 'k--', label=exp_label)
        axes[row_i, col_i].plot(test_angles, df_i[col] % 180 if col == "fitEllipse_theta_deg" else [v or 180 for v in df_i[col]],
                                 'ro-' if col_i == 0 else 'bs-', label=title)
        axes[row_i, col_i].set_title(f"{label} input: {title}")
        axes[row_i, col_i].legend(); axes[row_i, col_i].grid(True)
plt.tight_layout(); plt.savefig("orientation_part2_full_pipeline.png", dpi=150); plt.show()

---
## Part 3: Real boulder data

Runs the full orientation pipeline on real boulder mask polygons and plots the orientation distribution.

Two datasets are used:
- **Moon** — YOLO-predicted masks from `inference_fast3` (the pixelated masks we know are buggy)
- **Mars** — ground-truth manually digitized masks from Prieur et al. (2023), tiles from
  ESP_017355_2260, ESP_025650_2165, and ESP_028537_2270 (all at 0.25 m/px). These are smooth
  polygons, so they should show a correct/flat orientation distribution even before SAM2.

The Moon vs Mars comparison is deliberately apples-to-oranges in mask quality. That is the point:
YOLO spikes are a mask-quality problem — not a planet-specific problem.

In [ ]:
# ── Raster configuration ───────────────────────────────────────────────────
# Two modes:
#   "single-raster" (Moon): one large raster + YOLO inference output dir
#   "gt-tiles"      (Mars): many small GT shapefiles across train/val/test splits

home_p       = Path.home() / "Downloads" / "Test_ManualBoulderNetEnvironment"
work_dir     = home_p / "tmp" / "YOLOv8BeyondEarth"
raster_dir   = work_dir / "raster"
prieur_dir   = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px")

RASTER_CONFIGS = [
    {
        "label":      "Moon (YOLO)",
        "mode":       "single-raster",
        "in_raster":  raster_dir / "M139694087LE.tif",
        "output_dir": work_dir / "inference_fast3",
        "mask_glob":  "*-downscaled-mask-nms.shp",
        "res":        None,   # read from raster
    },
    {
        "label":      "Mars (GT, 0.25 m/px)",
        "mode":       "gt-tiles",
        "mask_dirs":  [
            prieur_dir / "preprocessing" / "train"      / "labels",
            prieur_dir / "preprocessing" / "validation" / "labels",
            prieur_dir / "preprocessing" / "test"       / "labels",
        ],
        # Only 0.25 m/px RED images (excludes the MRDR product at ~0.6 m/px)
        "mask_globs": [
            "ESP_017355_2260_RED_*_mask.shp",
            "ESP_025650_2165_RED_*_mask.shp",
            "ESP_028537_2270_RED_*_mask.shp",
        ],
        "res":        0.25,
    },
]

for cfg in RASTER_CONFIGS:
    if cfg["mode"] == "single-raster":
        status = "exists" if cfg["in_raster"].exists() else "NOT FOUND"
        print(f"{cfg['label']}: {cfg['in_raster'].name} — {status}")
    else:
        n_dirs = sum(1 for d in cfg["mask_dirs"] if d.exists())
        print(f"{cfg['label']}: {n_dirs}/{len(cfg['mask_dirs'])} label dirs found")

In [ ]:
df_per_raster    = {}
gdf_per_raster   = {}
res_per_raster   = {}
areal_per_raster = {}

for cfg in RASTER_CONFIGS:
    label = cfg["label"]
    res   = cfg.get("res")

    # ── Collect shapefiles ──────────────────────────────────────────────────
    if cfg["mode"] == "single-raster":
        in_raster = cfg["in_raster"]
        if not in_raster.exists():
            print(f"[{label}] Raster not found, skipping: {in_raster}")
            continue
        if res is None:
            res = raster_metadata.get_resolution(in_raster)[0]
        mask_shps = sorted(cfg["output_dir"].glob(cfg["mask_glob"]))
        if not mask_shps:
            print(f"[{label}] No shapefiles in {cfg['output_dir']} — run inference first.")
            continue

    else:  # gt-tiles mode
        mask_shps = []
        for d in cfg["mask_dirs"]:
            for g in cfg["mask_globs"]:
                mask_shps.extend(sorted(d.glob(g)))
        if not mask_shps:
            print(f"[{label}] No GT shapefiles found — check prieur_dir path.")
            continue

    areal_threshold = (res ** 2) * (4.74 ** 2)
    print(f"\n[{label}] res={res:.4f} m  areal_threshold={areal_threshold:.4f} m²  shapefiles={len(mask_shps)}")

    # ── Load & filter by area ───────────────────────────────────────────────
    gdfs     = [gpd.read_file(p) for p in mask_shps]
    gdf_all  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf_all["poly_area"] = gdf_all.geometry.area
    gdf_filt = gdf_all[gdf_all["poly_area"] >= areal_threshold].reset_index(drop=True)
    print(f"[{label}] Total: {len(gdf_all)}, after area filter: {len(gdf_filt)}")

    # ── Run orientation pipeline ────────────────────────────────────────────
    fit_failures = 0
    records = []
    for idx, row in tqdm(gdf_filt.iterrows(), total=len(gdf_filt), desc=label):
        poly     = row.geometry
        poly_seg = segmentize(poly, res)
        row_seg  = pd.Series({"geometry": poly_seg})
        try:
            ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
            fit_failed = False
        except Exception:
            ellipse_poly, a_fit, b_fit, theta_rad = poly, 1.0, 1.0, 1.0
            fit_failed = True
            fit_failures += 1
        theta_deg = np.degrees(theta_rad)
        try:
            mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
            (_, _, long_axis, short_axis, _, _, _, angle180) = boulder_row(mrr_row)
            aspect_ratio = long_axis / short_axis if short_axis > 0 else None
        except Exception:
            angle180, aspect_ratio = None, None
        records.append({
            "theta_deg":    theta_deg,
            "fit_failed":   fit_failed,
            "angle180":     angle180,
            "aspect_ratio": aspect_ratio,
            "poly_area":    row["poly_area"],
        })

    df = pd.DataFrame(records)
    print(f"[{label}] Fit failures: {fit_failures}/{len(gdf_filt)} ({100*fit_failures/len(gdf_filt):.1f}%)")
    df_per_raster[label]    = df
    gdf_per_raster[label]   = gdf_filt
    res_per_raster[label]   = res
    areal_per_raster[label] = areal_threshold

print("\nAvailable datasets:", list(df_per_raster.keys()))

# Backward-compat aliases for Part 5
if "Moon (YOLO)" in df_per_raster:
    df_real         = df_per_raster["Moon (YOLO)"]
    gdf_filtered    = gdf_per_raster["Moon (YOLO)"]
    res             = res_per_raster["Moon (YOLO)"]
    areal_threshold = areal_per_raster["Moon (YOLO)"]

In [ ]:
# Filter to boulders with aspect ratio 1.2–2.0 (the range of interest)
df_elongated = df_real[(df_real["aspect_ratio"] >= 1.2) & (df_real["aspect_ratio"] <= 2.0)]
print(f"Elongated boulders (1.2 ≤ a/b ≤ 2.0): {len(df_elongated)} / {len(df_real)}")
print(f"Of those, fit failures: {df_elongated['fit_failed'].sum()}")

# Check if theta=1.0 rad (57.3°) is overrepresented — the fallback value
print(f"\ntheta_deg value counts (top 10):")
print(df_elongated["theta_deg"].round(1).value_counts().head(10))

In [ ]:
for label, df in df_per_raster.items():
    df_elong = df[(df["aspect_ratio"] >= 1.2) & (df["aspect_ratio"] <= 2.0)]
    print(f"\n=== {label} ===")
    print(f"Elongated boulders (1.2 ≤ a/b ≤ 2.0): {len(df_elong)} / {len(df)}")
    print(f"Fit failures in elongated subset: {df_elong['fit_failed'].sum()}")
    print("theta_deg value counts (top 10):")
    print(df_elong["theta_deg"].round(1).value_counts().head(10))

n_rasters = len(df_per_raster)
fig, axes = plt.subplots(n_rasters, 3, figsize=(16, 5 * n_rasters))
if n_rasters == 1:
    axes = axes[np.newaxis, :]

for row_i, (label, df) in enumerate(df_per_raster.items()):
    df_elong = df[(df["aspect_ratio"] >= 1.2) & (df["aspect_ratio"] <= 2.0)]

    axes[row_i, 0].hist(df_elong["theta_deg"].dropna(), bins=36, range=(-180, 180),
                        color='tomato', edgecolor='k')
    axes[row_i, 0].axvline(np.degrees(1.0), color='navy', linestyle='--',
                            label=f'fallback = {np.degrees(1.0):.1f}°')
    axes[row_i, 0].set_title(f"[{label}] fitEllipse theta (elongated boulders)")
    axes[row_i, 0].set_xlabel("theta (°)")
    axes[row_i, 0].legend()

    axes[row_i, 1].hist(df_elong["angle180"].dropna(), bins=36, range=(0, 180),
                        color='steelblue', edgecolor='k')
    axes[row_i, 1].set_title(f"[{label}] Final angle180 (elongated boulders)")
    axes[row_i, 1].set_xlabel("angle180 (°)")

    axes[row_i, 2].hist(df["aspect_ratio"].dropna(), bins=40, color='seagreen', edgecolor='k')
    axes[row_i, 2].axvline(1.2, color='k', linestyle='--', label='1.2')
    axes[row_i, 2].axvline(2.0, color='k', linestyle=':', label='2.0')
    axes[row_i, 2].set_title(f"[{label}] Aspect ratio distribution (all boulders)")
    axes[row_i, 2].set_xlabel("a/b")
    axes[row_i, 2].legend()

plt.tight_layout()
plt.savefig("orientation_part3_real_data.png", dpi=150)
plt.show()
print("Plot saved to orientation_part3_real_data.png")

### Part 3 interpretation

**Moon (YOLO predictions):**
- Spikes at ~90° and ~180° in `theta_deg` → confirmed pixel-grid artifact from YOLO masks
- `angle180` also affected → biased orientation estimates for downstream science

**Mars (GT annotations, Prieur et al.):**
- `theta_deg` and `angle180` distributions should be substantially flatter / more uniform
- These are hand-digitized smooth polygons — no staircase artifact — so the pipeline recovers orientation correctly
- Any residual non-uniformity reflects true boulder orientation preferences on Mars, not an artifact

**Key takeaway:** The spikes are a *mask quality* problem, not a planet-specific one.
Running YOLO on Mars would produce the same spikes. SAM2-quality masks on Moon would be flat.
This motivates replacing YOLO's mask head with SAM2 for both datasets.

In [ ]:
SIZE_THRESHOLDS_M2 = [5.0, 10.0, 25.0, 50.0, 100.0]

for label, df in df_per_raster.items():
    at = areal_per_raster[label]
    thresholds = [(at, f"all (>{at:.1f} m²)")] + [(v, f">{v:.0f} m²") for v in SIZE_THRESHOLDS_M2]
    print(f"\n=== {label} ===")
    print(f"{'Threshold':>20}  {'n elongated':>12}  {'spike ~90°':>12}  {'spike ~180°':>12}")
    print("-" * 62)
    for min_area, lbl in thresholds:
        subset = df[
            (df["poly_area"] >= min_area) &
            (df["aspect_ratio"] >= 1.2) &
            (df["aspect_ratio"] <= 2.0)
        ]
        n = len(subset)
        spike_90  = ((subset["theta_deg"] > 80)  & (subset["theta_deg"] < 100)).sum()
        spike_180 = ((subset["theta_deg"] > 165) | (subset["theta_deg"] < -165)).sum()
        print(f"{lbl:>20}  {n:>12}  {spike_90:>12}  {spike_180:>12}")

In [ ]:
for label, df in df_per_raster.items():
    at = areal_per_raster[label]
    thresholds = [(at, f"all\n(>{at:.1f} m²)")] + [(v, f">{v:.0f} m²") for v in SIZE_THRESHOLDS_M2]
    n_col = len(thresholds)

    fig, axes = plt.subplots(2, n_col, figsize=(4 * n_col, 8))
    for col, (min_area, lbl) in enumerate(thresholds):
        subset = df[
            (df["poly_area"] >= min_area) &
            (df["aspect_ratio"] >= 1.2) &
            (df["aspect_ratio"] <= 2.0)
        ]
        axes[0, col].hist(subset["theta_deg"].dropna(), bins=36, range=(0, 180),
                          color='tomato', edgecolor='k')
        axes[0, col].set_title(f"theta\n{lbl}\n(n={len(subset)})")
        axes[0, col].set_xlabel("theta (°)")
        if col == 0:
            axes[0, col].set_ylabel("count")

        axes[1, col].hist(subset["angle180"].dropna(), bins=36, range=(0, 180),
                          color='steelblue', edgecolor='k')
        axes[1, col].set_title(f"angle180\n{lbl}\n(n={len(subset)})")
        axes[1, col].set_xlabel("angle180 (°)")
        if col == 0:
            axes[1, col].set_ylabel("count")

    plt.suptitle(
        f"[{label}] Orientation distribution by minimum boulder size\n"
        "Spikes at 90° and 180° should shrink as size increases if pixel-grid artifact is the cause",
        fontsize=12)
    plt.tight_layout()
    fname = f"orientation_part4_size_sweep_{label.lower()}.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Plot saved to {fname}")

### Part 4 interpretation

- **Spikes at 90°/180° shrink as threshold increases** → confirms the pixel-grid artifact hypothesis. Small boulders with boxy masks are the source; larger boulders have smoother outlines and more physically meaningful orientations.
- **Spikes persist even at large sizes** → the artifact is not purely a size issue — mask quality is poor even for larger boulders, likely because `downscale_pred=True` reduces effective pixel resolution for all sizes.
- **Distribution becomes more uniform at large sizes** → orientation is only reliable for larger boulders with this pipeline. SAM2 masks would extend reliable orientation estimation down to smaller boulders.

---
## Part 5: downscale_pred=False comparison

Re-runs detection on the same raster with `downscale_pred=False`, then compares
orientation histograms side by side against the original `downscale_pred=True` results.

**Prerequisite:** the detection model must already be loaded (`detection_model` in scope from
`example_of_tiled_predictions_fast.ipynb`), or re-load it in the cell below.

In [ ]:
import time
from YOLOv8BeyondEarth.predict import get_sliced_predictionfast

# Output dir for downscale_pred=False results
output_dir_nodl = work_dir / "inference_nodl"
output_dir_nodl.mkdir(parents=True, exist_ok=True)

# Load model
model_dir = work_dir / "yolov8_model"
model_weights = model_dir / "yolov8-m-boulder-detection-tmp.pt"
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=model_weights.as_posix(),
    confidence_threshold=0.1,
    device="cuda:0",
    image_size=1024)

# Run detection with downscale_pred=False — only slice_size=512 to match Part 3
print("Running detection with downscale_pred=False ...")
t0 = time.time()
get_sliced_predictionfast(
    in_raster,
    detection_model=detection_model,
    confidence_threshold=0.10,
    has_mask=True,
    output_dir=output_dir_nodl,
    interim_file_name=None,
    interim_dir=None,
    slice_size=512,
    inference_size=1024,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    min_area_threshold=6,
    downscale_pred=False,
    postprocess=True,
    postprocess_match_threshold=0.2,
    postprocess_class_agnostic=False,
    batch_size=16)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s  ({elapsed/60:.1f} min)")

In [ ]:
# Load and run orientation pipeline on downscale_pred=False shapefiles
mask_shps_nodl = sorted(output_dir_nodl.glob("*-mask-nms.shp"))
print(f"Found {len(mask_shps_nodl)} shapefile(s): {[p.name for p in mask_shps_nodl]}")

gdfs_nodl = [gpd.read_file(p) for p in mask_shps_nodl]
gdf_nodl = gpd.GeoDataFrame(pd.concat(gdfs_nodl, ignore_index=True), crs=gdfs_nodl[0].crs)
gdf_nodl["poly_area"] = gdf_nodl.geometry.area
gdf_nodl = gdf_nodl[gdf_nodl["poly_area"] >= areal_threshold].reset_index(drop=True)
print(f"Boulders after area filter: {len(gdf_nodl)}")

records_nodl = []
for idx, row in tqdm(gdf_nodl.iterrows(), total=len(gdf_nodl)):
    poly = row.geometry
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
    except Exception:
        ellipse_poly, a_fit, b_fit, theta_rad = poly, 1.0, 1.0, 1.0
    theta_deg = np.degrees(theta_rad)
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        (_, _, long_axis, short_axis, _, _, _, angle180) = boulder_row(mrr_row)
        aspect_ratio = long_axis / short_axis if short_axis > 0 else None
    except Exception:
        angle180, aspect_ratio = None, None
    records_nodl.append({"theta_deg": theta_deg, "angle180": angle180,
                          "aspect_ratio": aspect_ratio, "poly_area": row["poly_area"]})

df_nodl = pd.DataFrame(records_nodl)
print(df_nodl.describe())

In [ ]:
# Side-by-side comparison: downscale_pred=True vs False
# Filter to elongated boulders in both
elong_dl   = df_real[(df_real["aspect_ratio"] >= 1.2) & (df_real["aspect_ratio"] <= 2.0)]
elong_nodl = df_nodl[(df_nodl["aspect_ratio"] >= 1.2) & (df_nodl["aspect_ratio"] <= 2.0)]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, (df_e, label) in enumerate([(elong_dl, "downscale_pred=True"), (elong_nodl, "downscale_pred=False")]):
    axes[0, col].hist(df_e["theta_deg"].dropna(), bins=36, range=(0, 180),
                      color='tomato', edgecolor='k')
    axes[0, col].set_title(f"fitEllipse theta\n{label} (n={len(df_e)})")
    axes[0, col].set_xlabel("theta (°)")
    axes[0, col].set_ylabel("count")

    axes[1, col].hist(df_e["angle180"].dropna(), bins=36, range=(0, 180),
                      color='steelblue', edgecolor='k')
    axes[1, col].set_title(f"angle180\n{label} (n={len(df_e)})")
    axes[1, col].set_xlabel("angle180 (°)")
    axes[1, col].set_ylabel("count")

plt.suptitle("downscale_pred=True vs False — elongated boulders (1.2 ≤ a/b ≤ 2.0)\n"
             "Spikes at 90°/180° should shrink with downscale_pred=False", fontsize=12)
plt.tight_layout()
plt.savefig("orientation_part5_downscale_comparison.png", dpi=150)
plt.show()
print("Plot saved to orientation_part5_downscale_comparison.png")

---
## Part 6: Does the mask→polygon extraction step introduce the staircase?

**Question:** Is the orientation bug caused by the step where BoulderNet converts
YOLOv8's binary mask to a polygon, or is the staircase already baked into the binary mask?

**Design:**
1. Rasterize synthetic ellipses at known angles onto binary masks (simulating YOLO output)
2. Extract polygons via `CHAIN_APPROX_NONE` (current `binary_mask_to_polygon`) and `CHAIN_APPROX_SIMPLE` (alternative)
3. Run both through the full orientation pipeline (`segmentize → fitEllipse → MRR → boulder_row`)
4. Compare against the smooth polygon baseline (Part 2, confirmed correct)

**Expected result:** Smooth polygon tracks the expected angle perfectly;
both extraction methods give similar (bad) results — confirming the staircase
is inherent to the binary mask, not introduced by the extraction code.
No within-extraction fix is possible; SAM2 masks are the solution.

In [ ]:
import cv2

def rasterize_ellipse(a_px, b_px, theta_degrees, img_size=100):
    """Draw a filled ellipse centered in a binary uint8 mask.

    theta_degrees: CCW from x-axis (standard math convention, y-up).
    cv2.ellipse measures CW from x-axis in image coords (y-down), which is
    numerically equal to the same degree value for the visual orientation.
    """
    mask = np.zeros((img_size, img_size), dtype=np.uint8)
    cx, cy = img_size // 2, img_size // 2
    cv2.ellipse(mask, (cx, cy), (int(a_px), int(b_px)), float(theta_degrees), 0, 360, 255, -1)
    return mask


def mask_to_shapely(mask, approx_method):
    """Extract the largest contour from a uint8 binary mask and return a Shapely Polygon."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, approx_method)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea).squeeze(1)
    if len(contour) < 3:
        return None
    return Polygon(contour.astype(float))


# Three sizes representative of small / medium / large boulders in pixel space
p6_sizes = [
    (10,  7,   50,  "small  (a=10px, b=7px,  50×50 mask)"),
    (20,  13,  100, "medium (a=20px, b=13px, 100×100 mask)"),
    (40,  27,  200, "large  (a=40px, b=27px, 200×200 mask)"),
]

In [ ]:
p6_results = {}

for a_px, b_px, img_size, size_label in p6_sizes:
    records = []
    for deg in test_angles:
        rec = {"input_deg": deg}

        # (a) Smooth polygon baseline (same a/b in pixels, res=1.0 to match pixel units)
        poly_smooth = make_ellipse_polygon(a_px, b_px, deg, cx=img_size / 2, cy=img_size / 2)
        try:
            r = full_pipeline(poly_smooth, res=1.0)
            rec["smooth_theta"] = r["fitEllipse_theta_deg"] % 180
        except Exception as e:
            rec["smooth_theta"] = None
            print(f"smooth failed at {deg}°: {e}")

        # Rasterize to binary mask
        mask = rasterize_ellipse(a_px, b_px, deg, img_size=img_size)

        # (b) CHAIN_APPROX_NONE — current binary_mask_to_polygon behaviour
        poly_none = mask_to_shapely(mask, cv2.CHAIN_APPROX_NONE)
        if poly_none is not None:
            try:
                r = full_pipeline(poly_none, res=1.0)
                rec["none_theta"] = r["fitEllipse_theta_deg"] % 180
            except Exception as e:
                rec["none_theta"] = None
                print(f"NONE failed at {deg}°: {e}")
        else:
            rec["none_theta"] = None

        # (c) CHAIN_APPROX_SIMPLE — removes collinear points along straight edges
        poly_simple = mask_to_shapely(mask, cv2.CHAIN_APPROX_SIMPLE)
        if poly_simple is not None:
            try:
                r = full_pipeline(poly_simple, res=1.0)
                rec["simple_theta"] = r["fitEllipse_theta_deg"] % 180
            except Exception as e:
                rec["simple_theta"] = None
                print(f"SIMPLE failed at {deg}°: {e}")
        else:
            rec["simple_theta"] = None

        records.append(rec)

    p6_results[size_label] = pd.DataFrame(records)
    print(f"\n{size_label}")
    print(p6_results[size_label][["input_deg", "smooth_theta", "none_theta", "simple_theta"]].to_string())

In [ ]:
expected_mod180 = [d % 180 for d in test_angles]

fig, axes = plt.subplots(len(p6_sizes), 1, figsize=(10, 4 * len(p6_sizes)))
if len(p6_sizes) == 1:
    axes = [axes]

for ax, (size_label, df6) in zip(axes, p6_results.items()):
    ax.plot(test_angles, expected_mod180,      'k--', lw=2,  label='expected (input mod 180)')
    ax.plot(test_angles, df6["smooth_theta"],  'g^-', lw=1.5, label='smooth polygon (baseline)')
    ax.plot(test_angles, df6["none_theta"],    'ro-', lw=1.5, label='rasterized → CHAIN_APPROX_NONE')
    ax.plot(test_angles, df6["simple_theta"],  'bs--',lw=1.5, label='rasterized → CHAIN_APPROX_SIMPLE')
    ax.set_title(f"{size_label}")
    ax.set_xlabel("Input angle (°)")
    ax.set_ylabel("Recovered theta (°, mod 180)")
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True)
    ax.set_ylim(-5, 185)

plt.suptitle(
    "Part 6: Binary mask → polygon extraction\n"
    "Green (smooth) should track perfectly. Red ≈ Blue → extraction method is not the issue;\n"
    "deviation from expected confirms the staircase is in the binary mask itself.",
    fontsize=11)
plt.tight_layout()
plt.savefig("orientation_part6_rasterization_test.png", dpi=150)
plt.show()
print("Plot saved to orientation_part6_rasterization_test.png")

### Part 6 interpretation

- **Green tracks perfectly** → the downstream pipeline (`fitEllipse` → MRR → `boulder_row`) is correct; confirmed again as in Part 2.
- **Red ≈ Blue** → `CHAIN_APPROX_NONE` and `CHAIN_APPROX_SIMPLE` give nearly identical (bad) results. The choice of contour extraction method does **not** matter.
- **Red/Blue deviate from expected** → the staircase artifact is already present in the binary mask before any polygon extraction takes place. The extraction faithfully copies the rectilinear pixel boundary into the polygon — no extraction algorithm can invent smooth curves from a pixelated mask.
- **Larger masks (bottom panel) are closer to expected** → consistent with Part 4; more pixels per boulder means the staircase is a smaller fraction of the perimeter, so the ellipse fit is less dominated by horizontal/vertical edges.

**Conclusion:** The mask→polygon conversion step is **not** a bug and cannot be fixed within this step. The root cause is upstream: YOLOv8's binary mask representation. SAM2, which produces sub-pixel smooth instance masks, is the correct fix.